# FABS Track 2 — RL Maze Agent · Colab Trainer (v2)

End-to-end notebook: clones repo, generates maps, trains PPO 800k on T4, runs benchmark + ablation, downloads `viz/` and `agents/agent.zip`.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**Expected wall-clock on T4 free:**
- maps 30s
- train 800k ≈ 2–2.5 h
- benchmark 5 min
- ablation (4× 100k) ≈ 50 min (skip if short on time)

**If you've already cloned in this session,** skip Section 1 and start at Section 1b (`git pull`).

## 1. First-time setup — clone the repo

In [ ]:
!git clone https://github.com/Shalbulov/ai-hackathon-t2-maze-rl.git
%cd ai-hackathon-t2-maze-rl

## 1b. If repo already cloned — pull latest changes

In [ ]:
%cd /content/ai-hackathon-t2-maze-rl
!git pull

## 2. Install dependencies + verify GPU

Re-run this cell every time the Colab runtime reconnects (Colab wipes installed pip packages on disconnect).

In [ ]:
!pip install -q -r requirements.txt
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU not attached — Runtime → Change runtime type → T4 GPU'

## 3. Generate maps (4 train + 3 test, OOD-shifted)

In [ ]:
!python maze_gen.py --train 4 --test 3 --size 9 --seed 42
!ls maps/

## 4. Quick env smoke test (5 random steps)

Sanity check that env loads and observation/action spaces are correct.

In [ ]:
from env import Maze3DEnv
import numpy as np
env = Maze3DEnv('maps/train1.npy', randomize=True)
obs, info = env.reset(seed=0)
assert obs.shape == (23,), obs.shape
for _ in range(5):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
    print(f'step={env.steps} r={r:+.3f} pos={info["pos"]}')
print('OK')

## 5. Train PPO (2M steps, ~35 min on T4)

v4 setup:
- **BFS-distance reward shaping** — reward only fires for true progress along reachable paths (Euclidean shaping was rewarding wall-bashing).
- **Map shuffle** — every parallel env picks a random train map on each reset, forcing one general policy instead of 4 memorized ones.
- **Domain Randomization on** — random start cells across all train maps for exposure to diverse (map, start) configurations.

Watch `ep_rew_mean` and `success_rate` in the progress bar:

| Steps | `ep_rew_mean` | `success_rate` |
|---:|---:|---:|
| 100k | 5–15 | 0.3–0.5 |
| 500k | 30–45 | 0.7–0.85 |
| 1.5M | 50–60 | 0.9+ |
| 2M | **55–65** | **0.95+** |

If `success_rate < 0.5` at 500k → stop and ping for help.

In [ ]:
!python train.py --steps 2000000 --n-envs 4 --seed 42

## 6. Benchmark (100 ep × 7 maps + visualizations)

Loads `agents/agent.zip` (final) by default. To use the best checkpoint instead:
`!python benchmark.py --model agents/best_model.zip --episodes 100`

In [ ]:
!python benchmark.py --episodes 100

## 6b. Display results inline

In [ ]:
from IPython.display import Image, display
print(open('viz/results_table.txt').read())
print('\n--- before/after (random vs trained) ---')
display(Image('viz/before_after.gif'))
print('\n--- visit heatmap on train1 ---')
display(Image('viz/heatmap_train1.png'))
print('\n--- 3D trajectory (z = timestep) ---')
display(Image('viz/trajectory_3d.gif'))
print('\n--- learning curve ---')
display(Image('viz/learning_curve.png'))

## 7. Ablation study (~50 min, +8 rubric points — optional but high-value)

Trains 4 short variants (full / no_surface / no_progress / no_dr) at 100k steps each, then compares train/test step counts. Skip if running out of time — main agent is already saved.

In [ ]:
!python ablation.py --steps 100000 --episodes 20
print(open('viz/ablation.txt').read())

## 8. Download artifacts

Bundles `agents/agent.zip` + `viz/*` + `maps/*` into `submission.zip` and downloads it to your Mac.

In [ ]:
!zip -qr submission.zip agents/agent.zip agents/best_model.zip viz/ maps/ README.md 2>/dev/null
from google.colab import files
files.download('submission.zip')

## 9. (Highly recommended) Save to Google Drive — survives runtime disconnects

Run this **before** training kicks off (or right after) so a Colab disconnect doesn't lose 2 hours of GPU time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fabs_t2
!cp -r agents viz maps /content/drive/MyDrive/fabs_t2/ 2>/dev/null
!ls /content/drive/MyDrive/fabs_t2/

## Troubleshooting

**`ep_rew_mean` stuck near 0 / negative after 200k steps** → policy collapsed to "stand still". Stop training, lower `--n-envs` to 2, or raise `ent_coef` in `train.py`.

**`tostring_rgb` AttributeError** → outdated matplotlib in cached env. Run `!pip install -q --upgrade matplotlib` and re-run benchmark.

**`ale-py` install error** → already fixed in `requirements.txt` (no `[extra]`). Run `!git pull` and retry.

**Colab disconnects mid-training** → mount Drive (Section 9) BEFORE training. Resume from `agents/ppo_ckpt_*.zip` checkpoint by editing `train.py` to call `PPO.load(...)` instead of constructing fresh.

**Benchmark shows 250 steps / 0% success on most maps** → undertrained. Re-run train.py with `--steps 1500000` if T4 quota allows.